In [6]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path
import requests

In [2]:
def save_generated_annotation(events, prompt, file_path="generated_annotations_trials.txt"):

    file_path = Path(file_path)

    text = "\n".join(events) if isinstance(events, list) else str(events)

    record = {
        "timestamp_saved": datetime.now(timezone.utc).isoformat(),
        "prompt": prompt,
        "annotation": text
    }
    with file_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n\n")

    return record

In [3]:
def extract_events_with_ollama(prompt, model="llama3.2"):

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    
    return response.json()["response"]


def parse_salient_events(llm_output):
    events = []
    for line in llm_output.strip().splitlines():
        if not line.strip():
            continue

        # If line starts with "Salient event" keep text after first colon
        if ":" in line and line.lower().startswith("salient event"):
            description = line.split(":", 1)[1].strip()
        else:
            # remove leading "Salient event X" if present (with optional punctuation)
            description = re.sub(r"^\s*Salient event\s*\d+\s*[:\.\)\-]*\s*", "", line, flags=re.I)

            # remove leading numeric list markers like "1. ", "1) ", "1 - "
            description = re.sub(r"^\s*\d+\s*[:\.\)\-]*\s*", "", description)

        if description:
            events.append(description)
    return events

In [4]:
vlm_description = """The video starts with a person walking down a street in a city, passing by various buildings and shops. The scene then transitions to a person sitting on the ground, wearing earphones and looking at something in their hands. The next scene shows a group of people gathered around a banner, with one person holding a microphone. The video then shows a group of people standing on a beach, with a large building in the background. The final scene shows a person lying on a bed, holding a camera."""
prompt = f"""You are an expert video annotator. Below is a description of a video.
Understand the overall context and narrative of the video based on the description provided.
Extract between 4 and 10 salient events from this description.

A salient event is important for understanding the overall narrative, clearly distinguishable, and described in 1-2 concise sentences. Don't narrate the flow of the video, just describe the events.

Format EXACTLY like this:
Salient event 1: [description]
Salient event 2: [description]
...
Only output the numbered list, nothing else.

Video description:
{vlm_description}"""

In [7]:
result = extract_events_with_ollama(prompt)
print(result)

events = parse_salient_events(result)
print("\n--- Parsed Events ---")
for i, event in enumerate(events, 1):
    print(f"{event}")

save_generated_annotation(events, prompt=prompt)

1: A person is seen walking down a street in a city, passing by various buildings and shops.
2: A person sits on the ground, wearing earphones and looking at something in their hands.
3: A group of people gather around a banner with one person holding a microphone.
4: The scene cuts to a group of people standing on a beach, with a large building in the background.
5: A person lies on a bed, holding a camera.

--- Parsed Events ---
A person is seen walking down a street in a city, passing by various buildings and shops.
A person sits on the ground, wearing earphones and looking at something in their hands.
A group of people gather around a banner with one person holding a microphone.
The scene cuts to a group of people standing on a beach, with a large building in the background.
A person lies on a bed, holding a camera.


{'timestamp_saved': '2026-05-11T00:25:51.142270+00:00',
 'prompt': "You are an expert video annotator. Below is a description of a video.\nUnderstand the overall context and narrative of the video based on the description provided.\nExtract between 4 and 10 salient events from this description.\n\nA salient event is important for understanding the overall narrative, clearly distinguishable, and described in 1-2 concise sentences. Don't narrate the flow of the video, just describe the events.\n\nFormat EXACTLY like this:\nSalient event 1: [description]\nSalient event 2: [description]\n...\nOnly output the numbered list, nothing else.\n\nVideo description:\nThe video starts with a person walking down a street in a city, passing by various buildings and shops. The scene then transitions to a person sitting on the ground, wearing earphones and looking at something in their hands. The next scene shows a group of people gathered around a banner, with one person holding a microphone. The vide